# TP3 Recomendador personalizado de libros

## Paso 1 Cargar y revisar los libros

In [1]:
import pandas as pd
import os
file_path = 'catalogo_tp3.jsonl'
df = pd.read_json(file_path, lines=True)


In [2]:
# Vemos información general sobre las columnas y tipos de datos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8747 entries, 0 to 8746
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   source_url      8747 non-null   str                
 1   slug            8747 non-null   str                
 2   title           8747 non-null   str                
 3   authors         8747 non-null   object             
 4   genres          8747 non-null   object             
 5   seed_genre      8747 non-null   str                
 6   series          2316 non-null   str                
 7   series_number   2298 non-null   float64            
 8   cover_url       8747 non-null   str                
 9   synopsis        8746 non-null   str                
 10  synopsis_chars  8747 non-null   int64              
 11  scraped_at      8747 non-null   datetime64[us, UTC]
dtypes: datetime64[us, UTC](1), float64(1), int64(1), object(2), str(7)
memory usage: 820.2+ KB


In [3]:
# Conservamos solo las columnas requeridas
columnas_a_mantener = ['slug', 'title', 'authors', 'synopsis']
df_filtrado = df[columnas_a_mantener].copy()

# ponemos como id la columna slug? o solo identificamos el libro con ese titulo

# Eliminamos los registros sin sinopsis
df_filtrado = df_filtrado.dropna(subset=['synopsis'])
df_filtrado = df_filtrado[df_filtrado['synopsis'].str.strip() != '']

# Eliminamos duplicados por el identificador 'slug'
df_filtrado = df_filtrado.drop_duplicates(subset=['slug'])

# Mostramos la cantidad de registros restantes y una vista previa
print(f"Cantidad de registros restantes: {len(df_filtrado)}")
display(df_filtrado.head())

Cantidad de registros restantes: 8746


,slug,title,authors,synopsis
0,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],La Humanidad ha entrado en una nueva fase evol...
1,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],La extracción de ADN de los huesos de nuestros...
2,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],Teotihuacan es la primera gran ciudad del cent...
3,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],Entre los siglos VIII y VI a. C. los fenicios ...
4,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],Las raíces evolutivas del sufrimiento humano\n...


In [4]:
import re

# Normalizamos espacios en blanco
def normalizar_espacios(texto):
    if isinstance(texto, str):
        # \s+ coincide con cualquier espacio en blanco, incluidos \n, \r, \t, y reemplaza por uno unico
        return re.sub(r'\s+', ' ', texto).strip()
    return texto

# Aplicamos la funcion para normalizar las columnas de texto relevantes
columnas_texto = ['title', 'authors', 'synopsis']
for col in columnas_texto:
    df_filtrado[col] = df_filtrado[col].apply(normalizar_espacios)

# Mostramos para verificar los resultados
display(df_filtrado.head())

,slug,title,authors,synopsis
0,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],La Humanidad ha entrado en una nueva fase evol...
1,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],La extracción de ADN de los huesos de nuestros...
2,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],Teotihuacan es la primera gran ciudad del cent...
3,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],Entre los siglos VIII y VI a. C. los fenicios ...
4,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],Las raíces evolutivas del sufrimiento humano U...


In [5]:
# Contar la cantidad de registros antes de eliminar duplicados de sinopsis
total_antes = len(df_filtrado)

# Identificar y eliminar duplicados basados únicamente en la columna 'synopsis'
# 'keep="first"' conserva el primer libro que aparece con esa sinopsis
df_filtrado = df_filtrado.drop_duplicates(subset=['synopsis'], keep='first')

# Calcular cuántos registros se eliminaron
total_despues = len(df_filtrado)
eliminados = total_antes - total_despues

print(f"Cantidad de registros antes: {total_antes}")
print(f"Cantidad de registros después de eliminar sinopsis duplicadas: {total_despues}")
print(f"Se eliminaron {eliminados} libros con sinopsis repetidas.")

Cantidad de registros antes: 8746
Cantidad de registros después de eliminar sinopsis duplicadas: 8648
Se eliminaron 98 libros con sinopsis repetidas.


In [6]:
!pip install thefuzz

In [7]:
from thefuzz import fuzz
import pandas as pd

# Convertiremos los autores a string para agrupar de forma limpia
df_filtrado['authors_str'] = df_filtrado['authors'].apply(lambda x: " ".join(x) if isinstance(x, list) else str(x))

# Listas para guardar los ejemplos de unir y separar
ejemplos_unidos = []
ejemplos_separados = []

# Conjunto para rastrear los índices de registros a eliminar
indices_a_eliminar = set()

# Agrupamos por autor para evitar la comparación de todos contra todos
grupos = df_filtrado.groupby('authors_str')

for autor, grupo in grupos:
    if len(grupo) < 2:
        continue

    # Convertimos el grupo a una lista de diccionarios para iterar fácilmente
    libros = grupo.to_dict('records')
    n = len(libros)

    for i in range(n):
        idx_i = grupo.index[i]
        if idx_i in indices_a_eliminar:
            continue

        for j in range(i + 1, n):
            idx_j = grupo.index[j]
            if idx_j in indices_a_eliminar:
                continue

            titulo_i = libros[i]['title']
            titulo_j = libros[j]['title']
            sinopsis_i = libros[i]['synopsis']
            sinopsis_j = libros[j]['synopsis']

            # Calculamos similitudes
            sim_titulo = fuzz.token_sort_ratio(titulo_i.lower(), titulo_j.lower())
            sim_sinopsis = fuzz.token_sort_ratio(sinopsis_i.lower(), sinopsis_j.lower())

            # Criterio:
            # 1. Si los títulos son casi idénticos (ej. > 95%)
            # 2. Si los títulos son muy similares (ej. > 80%) Y la sinopsis tiene similitud moderada (ej. > 50%)
            deben_unirse = (sim_titulo > 95) or (sim_titulo > 80 and sim_sinopsis > 50)

            # Evitar unir obras numeradas de forma evidente (como Tomo I vs Tomo II)
            # Si el título contiene números romanos o dígitos diferentes al final, no los unimos
            es_secuela = False
            palabras_i = set(titulo_i.lower().split())
            palabras_j = set(titulo_j.lower().split())
            diferencia = palabras_i.symmetric_difference(palabras_j)
            secuela_keywords = {'i', 'ii', 'iii', 'iv', 'v', 'vi', '1', '2', '3', '4', '5'}
            if diferencia.intersection(secuela_keywords):
                es_secuela = True

            if deben_unirse and not es_secuela:
                indices_a_eliminar.add(idx_j)
                if len(ejemplos_unidos) < 2:
                    ejemplos_unidos.append((
                        (titulo_i, sinopsis_i[:150] + "..."),
                        (titulo_j, sinopsis_j[:150] + "..."),
                        f"Sim. Título: {sim_titulo}%, Sim. Sinopsis: {sim_sinopsis}%"
                    ))
            else:
                if len(ejemplos_separados) < 2 and (sim_titulo > 70):
                    ejemplos_separados.append((
                        (titulo_i, sinopsis_i[:150] + "..."),
                        (titulo_j, sinopsis_j[:150] + "..."),
                        f"Sim. Título: {sim_titulo}%, Sim. Sinopsis: {sim_sinopsis}%"
                    ))

# Creamos el dataframe final sin las obras consideradas duplicadas/versiones casi iguales
df_final = df_filtrado.drop(index=list(indices_a_eliminar))

# Imprimimos resultados
print(f"Registros antes de remover versiones casi iguales: {len(df_filtrado)}")
print(f"Registros actuales: {len(df_final)}")
print(f"Se removieron {len(indices_a_eliminar)} versiones duplicadas.\n")

Registros antes de remover versiones casi iguales: 8648
Registros actuales: 8575
Se removieron 73 versiones duplicadas.



In [8]:
print("=== DOS PARES QUE DECIDIMOS UNIR ===")
for idx, (obra1, obra2, razon) in enumerate(ejemplos_unidos, 1):
    print(f"\nPar {idx} ({razon}):")
    print(f"  - Obra A: \"{obra1[0]}\" | Sinopsis: {obra1[1]}")
    print(f"  - Obra B: \"{obra2[0]}\" | Sinopsis: {obra2[1]}")

print("\n=== DOS PARES QUE DECIDIMOS SEPARAR ===")
for idx, (obra1, obra2, razon) in enumerate(ejemplos_separados, 1):
    print(f"\nPar {idx} ({razon}):")
    print(f"  - Obra A: \"{obra1[0]}\" | Sinopsis: {obra1[1]}")
    print(f"  - Obra B: \"{obra2[0]}\" | Sinopsis: {obra2[1]}")

=== DOS PARES QUE DECIDIMOS UNIR ===

Par 1 (Sim. Título: 100%, Sim. Sinopsis: 55%):
  - Obra A: "Nuevos cuentos de Bustos Domecq" | Sinopsis: Caso ejemplar y a la vez rarísimo de escritura en colaboración, H. Bustos Domecq no es, sin embargo, la suma de sus dos ilustres creadores ni es asimi...
  - Obra B: "Nuevos cuentos de Bustos Domecq" | Sinopsis: Hace cincuenta años que H. Bustos Domecq ocupa un puesto de honor en el poco dilatado territorio de la literatura argentina. Nacido de la amistad entr...

Par 2 (Sim. Título: 91%, Sim. Sinopsis: 57%):
  - Obra A: "Historias negras del fútbol argentino" | Sinopsis: Hinchas que pretenden ahorcar a un árbitro, dirigentes que practican el soborno a repetición, maniobras para evitar descensos, sobre todo cuando afect...
  - Obra B: "Nuevas historias negras del fútbol argentino" | Sinopsis: Denuncias por sobornos, partidos con resultados mentirosos, otros consensuados a tal punto que el jugador que mete un gol, en vez de gritarlo, se agar...



In [9]:
# 1. Mostrar 5 registros de muestra del catálogo final limpio
print("=== 5 REGISTROS DE MUESTRA DEL CATÁLOGO LIMPIO ===")
display(df_final.head(5))

# 2. Calcular estadísticas de eliminación paso a paso
registros_iniciales = 8747

# Paso 1: Sinopsis vacías/nulas y duplicados de slug
# Inicialmente: 8747
# Después de dropna y slug deduplication: 8746 (se eliminaron 1)
sinopsis_vacias_o_duplicados_slug = registros_iniciales - 8746

# Paso 2: Sinopsis idénticas
# De 8746 a 8648 (se eliminaron 98)
eliminados_sinopsis_idencitas = 8746 - 8648

# Paso 3: Versiones casi iguales del mismo autor
# De 8648 a 8575 (se eliminaron 73)
eliminados_versiones_casi_iguales = 8648 - 8575

total_disponibles = len(df_final)
total_eliminados = registros_iniciales - total_disponibles

# 3. Mostramos resumen del proceso
print("\n=== RESUMEN DEL PROCESO DE LIMPIEZA ===")
print(f"Libros disponibles al final: {total_disponibles}")
print(f"Total de libros eliminados: {total_eliminados}")
print(f"  - Por sinopsis vacías o identificador (slug) duplicado: {sinopsis_vacias_o_duplicados_slug}")
print(f"  - Por sinopsis exactamente idénticas (mismo curso/serie): {eliminados_sinopsis_idencitas}")
print(f"  - Por versiones casi iguales del mismo autor (mismo título/sinopsis similar): {eliminados_versiones_casi_iguales}")

=== 5 REGISTROS DE MUESTRA DEL CATÁLOGO LIMPIO ===


,slug,title,authors,synopsis,authors_str
0,de-la-caverna-al-cosmos,De la caverna al cosmos,[Eudald Carbonell],La Humanidad ha entrado en una nueva fase evol...,Eudald Carbonell
1,quienes-somos-y-como-hemos-llegado-hasta-aqui,Quiénes somos y cómo hemos llegado hasta aquí,[David Reich],La extracción de ADN de los huesos de nuestros...,David Reich
2,teotihuacan,Teotihuacan,[Eduardo Matos Moctezuma],Teotihuacan es la primera gran ciudad del cent...,Eduardo Matos Moctezuma
3,tiro-y-las-colonias-fenicias-de-occidente,Tiro y las colonias fenicias de Occidente,[M.ª Eugenia Aubet],Entre los siglos VIII y VI a. C. los fenicios ...,M.ª Eugenia Aubet
4,la-maldicion-del-hombre-mono,La maldición del hombre mono,[Emiliano Bruner],Las raíces evolutivas del sufrimiento humano U...,Emiliano Bruner



=== RESUMEN DEL PROCESO DE LIMPIEZA ===
Libros disponibles al final: 8575
Total de libros eliminados: 172
  - Por sinopsis vacías o identificador (slug) duplicado: 1
  - Por sinopsis exactamente idénticas (mismo curso/serie): 98
  - Por versiones casi iguales del mismo autor (mismo título/sinopsis similar): 73


Al terminar el paso 1 quedan **8575 libros** disponibles. Se eliminaron 172: 1 sin sinopsis, 98 por sinopsis exactamente repetidas (cursos y series que repiten el mismo texto) y 73 por ser versiones casi iguales de otra obra del mismo autor.

## Paso 2 Auditar el catálogo

Dividimos `muestra_etiquetada.csv` por libro: 70 % para desarrollar los detectores y 30 % para medirlos. Toda decisión (umbrales, expresiones, criterios) se toma mirando solo la parte de desarrollo; la parte de prueba se usa únicamente para medir.

In [10]:
!pip install langdetect sentence-transformers
!python -m spacy download es_core_news_lg

In [11]:
import re
import unicodedata
import numpy as np
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score

muestra = pd.read_csv('muestra_etiquetada.csv', keep_default_na=False)

# Normalizamos los espacios igual que en el catálogo
muestra['synopsis'] = muestra['synopsis'].apply(normalizar_espacios)
muestra['fragmento'] = muestra['fragmento'].apply(normalizar_espacios)

desarrollo, prueba = train_test_split(
    muestra,
    test_size=0.3,
    random_state=42,
    stratify=muestra['contaminado']
)
print(len(desarrollo), len(prueba))
print(f"Contaminados en desarrollo: {desarrollo['contaminado'].sum()} | en prueba: {prueba['contaminado'].sum()}")

350 150
Contaminados en desarrollo: 70 | en prueba: 30


In [12]:
def oraciones(texto):
    partes = re.split(r"(?<=[.!?])\s+|\n", texto)
    return [p.strip() for p in partes if len(p.strip()) >= 15]

# Guardamos la precisión y el recall de cada detector para la tabla final
resultados = {}

def medir(nombre, marcado, datos):
    y = datos['contaminado']
    p = precision_score(y, marcado, zero_division=0)
    r = recall_score(y, marcado)
    resultados[nombre] = {'precision': round(p, 2), 'recall': round(r, 2), 'marcados': int(np.sum(marcado))}
    print(f"{nombre}: precisión {p:.2f}, recall {r:.2f} ({int(np.sum(marcado))} libros marcados)")

### 2.1 Expresiones regulares

Una sola expresión con tres partes: URLs (con `http`, con `www.` o con un nombre de dominio), correos y teléfonos con prefijo internacional. Para los dominios usamos una lista de terminaciones: las comunes (`.com`, `.net`, `.org`...) y las reservadas (`.example`, `.invalid`, `.test`) que aparecen en los fragmentos de desarrollo. El teléfono exige el `+` inicial, porque sin él la regex marcaba rangos de años como «1914-1918».

In [13]:
URL = r"https?://|www\.|\b[\w-]+\.(?:com|net|org|info|io|example|invalid|test)\b"
EMAIL = r"[\w.+-]+@[\w-]+\.[\w.-]+"
TELEFONO = r"\+\d{1,3}[\s-]?\d[\d\s-]{6,}\d"
REGEX = re.compile(f"{URL}|{EMAIL}|{TELEFONO}", re.IGNORECASE)

def detector_regex(texto):
    return bool(REGEX.search(texto))

medir("regex mínima", prueba['synopsis'].str.contains(re.compile(r"https?://|www\.", re.IGNORECASE)), prueba)
medir("regex", prueba['synopsis'].apply(detector_regex), prueba)

regex mínima: precisión 1.00, recall 0.13 (4 libros marcados)
regex: precisión 1.00, recall 0.50 (15 libros marcados)


In [14]:
# Revisamos en desarrollo los falsos positivos y los contaminados que la regex no encuentra
marca_dev = desarrollo['synopsis'].apply(detector_regex)

print("=== Falsos positivos (sinopsis limpias marcadas) ===")
for _, fila in desarrollo[marca_dev & (desarrollo['contaminado'] == 0)].iterrows():
    print(f"  {fila['slug']}: {REGEX.findall(fila['synopsis'])}")

print("\n=== Fragmentos que la regex no detecta (primeros 12) ===")
for f in desarrollo[~marca_dev & (desarrollo['contaminado'] == 1)]['fragmento'].head(12):
    print("  -", f[:110])

=== Falsos positivos (sinopsis limpias marcadas) ===
  que-pasaria-si: ['http://', 'www.', 'xkcd.com']

=== Fragmentos que la regex no detecta (primeros 12) ===
  - El libro más esperado del año llega por fin en edición digital completa. Leé este y miles de libros más sin pu
  - Conseguí este título y más de 50.000 libros en la app LetraTotal: el primer mes es sin cargo.
  - Si te gustó esta reseña, compartila con tus amigos y participá del sorteo de un lector electrónico.
  - Ganá dinero desde tu casa recomendando libros. Cupos limitados, consultá requisitos.
  - Leé este y miles de libros más sin publicidad. Activá tu prueba gratuita desde tu celular.
  - Si te gustó esta reseña, compartila con tus amigos y participá del sorteo de un lector electrónico.
  - Ganá dinero desde tu casa recomendando libros. Cupos limitados, consultá requisitos.
  - ¡Oferta por tiempo limitado! Suscribite hoy y llevate tres libros de regalo.
  - Link en nuestro sitio: h t t p s : / / epubhub [.] example
 

**Resultados 2.1.** La regex mínima del enunciado tiene precisión 1.00 y recall 0.13 en prueba. Agregando dominios, correos y teléfonos llega a precisión 1.00 y recall 0.50.

En desarrollo hay un solo falso positivo: *que-pasaria-si*, una sinopsis legítima que menciona `http://www.xkcd.com`, el sitio real del autor. Por eso una URL sola no prueba que haya spam. En prueba no aparecieron casos así, pero en el catálogo completo puede haber más.

- **Detecta bien:** la contaminación con enlaces escritos de forma literal (`https://...`, `www....`, `sitio.test`), los correos (`ventas@...`) y los teléfonos de WhatsApp con prefijo `+54`.
- **No detecta:** los enlaces disimulados («h t t p s : / / tomoya [.] test», «TomoClub punto test barra blasfemia», letras cirílicas o invisibles dentro del dominio) y todo el texto promocional sin enlace («Ganá dinero desde tu casa...», «Si te gustó esta reseña, compartila...», «Conseguí este título... en la app LetraTotal»).

### 2.2 Normalizar antes de buscar

Con `unicodedata` revisamos qué caracteres raros aparecen en los fragmentos de desarrollo y cuáles en las sinopsis limpias.

In [15]:
def caracteres_raros(textos):
    conteo = Counter()
    for t in textos:
        for ch in t:
            nombre = unicodedata.name(ch, '?')
            if unicodedata.category(ch) == 'Cf' or nombre.startswith(('CYRILLIC', 'GREEK')):
                conteo[(repr(ch), nombre, unicodedata.category(ch))] += 1
    return conteo

print("Fragmentos contaminados:")
for k, v in caracteres_raros(desarrollo['fragmento']).most_common():
    print(f"  {k} -> {v}")
print("\nSinopsis limpias:")
for k, v in caracteres_raros(desarrollo[desarrollo['contaminado'] == 0]['synopsis']).most_common():
    print(f"  {k} -> {v}")

Fragmentos contaminados:
  ("'\\u200b'", 'ZERO WIDTH SPACE', 'Cf') -> 4
  ("'\\u2060'", 'WORD JOINER', 'Cf') -> 3
  ("'а'", 'CYRILLIC SMALL LETTER A', 'Ll') -> 2
  ("'і'", 'CYRILLIC SMALL LETTER BYELORUSSIAN-UKRAINIAN I', 'Ll') -> 2
  ("'е'", 'CYRILLIC SMALL LETTER IE', 'Ll') -> 2
  ("'о'", 'CYRILLIC SMALL LETTER O', 'Ll') -> 1
  ("'с'", 'CYRILLIC SMALL LETTER ES', 'Ll') -> 1

Sinopsis limpias:
  ("'\\u2060'", 'WORD JOINER', 'Cf') -> 6
  ("'\\xad'", 'SOFT HYPHEN', 'Cf') -> 5


In [16]:
# Letras cirílicas que se ven iguales a las latinas
HOMOGLIFOS = {
    'а': 'a', 'е': 'e', 'о': 'o', 'р': 'p', 'с': 'c', 'х': 'x', 'у': 'y',
    'і': 'i', 'ј': 'j', 'ѕ': 's', 'ԁ': 'd',
    'А': 'A', 'Е': 'E', 'О': 'O', 'Р': 'P', 'С': 'C', 'Х': 'X', 'Т': 'T',
    'М': 'M', 'Н': 'H', 'К': 'K', 'В': 'B',
}
TERMINACIONES = r"(?:com|net|org|info|io|example|invalid|test)\b"

def normalizar(texto):
    # Quitamos los caracteres invisibles (categoría Cf: zero width space, word joiner, soft hyphen)
    texto = ''.join(ch for ch in texto if unicodedata.category(ch) != 'Cf')
    # Reemplazamos las letras de otro alfabeto por su equivalente latina
    texto = ''.join(HOMOGLIFOS.get(ch, ch) for ch in texto)
    # Signos escritos con palabras o entre corchetes
    texto = re.sub(r"\s*\[\.\]\s*", ".", texto)
    texto = re.sub(r"\s*[(\[](?:punto|dot)[)\]]\s*", ".", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\s+punto\s+(?=" + TERMINACIONES + ")", ".", texto, flags=re.IGNORECASE)
    texto = re.sub(r"(\." + TERMINACIONES + r")\s+barra\s+", r"\1/", texto, flags=re.IGNORECASE)
    texto = re.sub(r"\s+arroba\s+", "@", texto, flags=re.IGNORECASE)
    # Letras separadas por espacios: "w w w" -> "www"
    texto = re.sub(r"\b(?:\w ){2,}\w\b", lambda m: m.group(0).replace(' ', ''), texto)
    # Espacios alrededor de los signos de una dirección: "https : / / sitio . test"
    texto = re.sub(r"\s*:\s*/\s*/\s*", "://", texto)
    texto = re.sub(r"\s+\.\s+", ".", texto)
    return texto

def detector_regex_norm(texto):
    return detector_regex(normalizar(texto))

medir("regex + normalización", prueba['synopsis'].apply(detector_regex_norm), prueba)

regex + normalización: precisión 1.00, recall 0.63 (19 libros marcados)


In [17]:
# Casos que la normalización recuperó en desarrollo
recuperados = desarrollo[~desarrollo['synopsis'].apply(detector_regex) & desarrollo['synopsis'].apply(detector_regex_norm)]
print(f"Libros de desarrollo recuperados por la normalización: {len(recuperados)}")
print(f"Contaminados entre ellos: {recuperados['contaminado'].sum()}\n")
for _, fila in recuperados[recuperados['contaminado'] == 1].head(2).iterrows():
    print(f"Fragmento original:   {fila['fragmento']!r}")
    print(f"Fragmento normalizado: {normalizar(fila['fragmento'])!r}")
    print(f"Coincidencia:         {REGEX.findall(normalizar(fila['fragmento']))}\n")

# Los invisibles solos no alcanzan: también aparecen en sinopsis limpias
con_invisibles = desarrollo['synopsis'].apply(lambda t: any(unicodedata.category(ch) == 'Cf' for ch in t))
print(pd.crosstab(desarrollo['contaminado'], con_invisibles, rownames=['contaminado'], colnames=['tiene invisibles']))

Libros de desarrollo recuperados por la normalización: 10
Contaminados entre ellos: 10

Fragmento original:   'Link en nuestro sitio: h t t p s : / / epubhub [.] example'
Fragmento normalizado: 'Link en nuestro sitio: https://epubhub.example'
Coincidencia:         ['https://', 'epubhub.example']

Fragmento original:   'Link en nuestro sitio: h t t p s : / / lectorplus [.] invalid'
Fragmento normalizado: 'Link en nuestro sitio: https://lectorplus.invalid'
Coincidencia:         ['https://', 'lectorplus.invalid']



tiene invisibles  False  True 
contaminado                   
0                   276      4
1                    63      7


**Resultados 2.2.** Con la normalización el recall sube de 0.50 a 0.63 y la precisión sigue en 1.00. En desarrollo la normalización recuperó 10 libros nuevos y los 10 eran contaminados. Arriba se muestran dos casos recuperados: «h t t p s : / / epubhub [.] example» pasa a `https://epubhub.example`, y lo mismo pasa con `lectorplus.invalid`.

`unicodedata` muestra que los fragmentos usan *ZERO WIDTH SPACE* y *WORD JOINER* (categoría `Cf`) y letras cirílicas (*CYRILLIC SMALL LETTER A, IE, O, ES, BYELORUSSIAN-UKRAINIAN I*). Las sinopsis limpias también tienen caracteres `Cf` (*WORD JOINER*, *SOFT HYPHEN*): 4 de los 280 libros limpios de desarrollo los tienen. Por eso la función los elimina, pero no los toma como evidencia por sí solos: solo marcamos el libro si después de normalizar aparece una dirección.

### 2.3 Detectar el idioma de cada oración

Primero detectamos el idioma de la sinopsis completa.

In [18]:
from langdetect import detect, detect_langs, DetectorFactory
DetectorFactory.seed = 0

def idioma(texto):
    try:
        return detect(texto)
    except Exception:
        return 'desconocido'

idioma_sinopsis = desarrollo['synopsis'].apply(idioma)
print(pd.crosstab(desarrollo['contaminado'], idioma_sinopsis, rownames=['contaminado'], colnames=['idioma sinopsis']))

idioma sinopsis   es
contaminado         
0                280
1                 70


La sinopsis completa sale en español en casi todos los contaminados: el mensaje insertado es una oración entre varias, y el texto en español domina. Por eso pasamos a detectar el idioma oración por oración. Calculamos `detect_langs` una vez por oración y probamos distintas combinaciones de cantidad mínima de palabras y umbral de probabilidad en desarrollo.

In [19]:
def idiomas_oraciones(texto):
    res = []
    for o in oraciones(texto):
        try:
            mejor = detect_langs(o)[0]
        except Exception:
            continue
        res.append((len(o.split()), mejor.lang, mejor.prob, o))
    return res

idiomas_dev = desarrollo['synopsis'].apply(idiomas_oraciones)

def marca_idioma(lista, min_palabras, umbral):
    return any(n >= min_palabras and lang != 'es' and prob >= umbral for n, lang, prob, _ in lista)

filas = []
for min_palabras in [3, 5, 7, 10]:
    for umbral in [0.80, 0.90, 0.95, 0.99]:
        marcado = idiomas_dev.apply(lambda l: marca_idioma(l, min_palabras, umbral))
        y = desarrollo['contaminado']
        filas.append({'min_palabras': min_palabras, 'umbral': umbral,
                      'marcados': int(marcado.sum()),
                      'precision': round(precision_score(y, marcado, zero_division=0), 2),
                      'recall': round(recall_score(y, marcado), 2)})
grilla_idioma = pd.DataFrame(filas)
display(grilla_idioma)

,min_palabras,umbral,marcados,precision,recall
0,3,0.80,31,0.52,0.23
1,3,0.90,27,0.56,0.21
2,3,0.95,27,0.56,0.21
3,3,0.99,27,0.56,0.21
4,5,0.80,23,0.61,0.20
5,5,0.90,23,0.61,0.20
6,5,0.95,23,0.61,0.20
7,5,0.99,23,0.61,0.20
8,7,0.80,17,0.76,0.19
9,7,0.90,15,0.80,0.17


In [20]:
# Qué oraciones limpias se detectan en otro idioma (con la configuración más permisiva)
errores = Counter()
ejemplos = []
for lista, cont in zip(idiomas_dev, desarrollo['contaminado']):
    if cont == 0:
        for n, lang, prob, o in lista:
            if lang != 'es' and n >= 3:
                errores[lang] += 1
                ejemplos.append((n, lang, round(prob, 2), o[:90]))
print(errores.most_common())
for e in sorted(ejemplos, key=lambda x: -x[2])[:10]:
    print(" ", e)

[('pt', 11), ('ca', 4), ('fr', 3), ('en', 2), ('it', 2), ('de', 1), ('fi', 1)]
  (12, 'ca', 1.0, 'Es ésta la segunda entrega del ciclo que se inicia con Redwall.')
  (5, 'pt', 1.0, '¿Por qué existe este caos?')
  (5, 'pt', 1.0, '¿LA MAGIA O LA ESPADA?')
  (6, 'ca', 1.0, 'Octava aventura de Duke, de J.')
  (6, 'pt', 1.0, '¿Qué narra el mito de Osiris?')
  (8, 'pt', 1.0, 'A través de ¿Cómo pensar como Sherlock Holmes?')
  (3, 'pt', 1.0, '«Una obra maestra.')
  (5, 'fr', 1.0, '«Mi nombre es Jean Moreau.')
  (3, 'pt', 1.0, '¿Por qué descender?')
  (7, 'pt', 1.0, '¿ESTÁS DISPUESTO A PLANTAR CARA AL PELIGRO?')


In [21]:
MIN_PALABRAS = 10
UMBRAL_IDIOMA = 0.90

def detector_idioma(texto):
    return marca_idioma(idiomas_oraciones(texto), MIN_PALABRAS, UMBRAL_IDIOMA)

medir("idioma por oración", prueba['synopsis'].apply(detector_idioma), prueba)

idioma por oración: precisión 0.89, recall 0.27 (9 libros marcados)


**Resultados 2.3.** Con la sinopsis completa, `langdetect` da español en los 350 libros de desarrollo, incluidos los 70 contaminados. El mensaje en inglés o portugués es una oración dentro de un texto largo en español y no cambia el resultado. A nivel de sinopsis el detector no ve nada.

Oración por oración aparecen dos problemas, que se ven en la grilla y en la lista de errores de las sinopsis limpias:

- **Umbral de probabilidad:** casi no cambia los resultados. `detect_langs` devuelve probabilidad 1.0 incluso cuando se equivoca («¿Por qué existe este caos?» → `pt` con 1.0; «Es ésta la segunda entrega del ciclo...» → `ca` con 1.0). Elegimos **0.90** para descartar los casos dudosos, pero lo que importa es el largo.
- **Cantidad mínima de palabras:** es lo que decide. Los errores en sinopsis limpias son preguntas cortas, citas entre comillas y oraciones con nombres propios («Octava aventura de Duke, de J.»). La precisión en desarrollo pasa de 0.52 con 3 palabras a 0.89 con 10. Los mensajes reales en otro idioma tienen entre 10 y 14 palabras («Download this ebook for FREE, no sign-up required, at ... today.», «Leia milhares de livros sem pagar nada...»), así que con **10 palabras** no perdemos ninguno. Las oraciones cortas no se pueden evaluar con confianza y las dejamos afuera.

En prueba: precisión 0.89 y recall 0.27. El recall es bajo porque solo una parte de la contaminación está en otro idioma, pero en ese tipo detecta el 100 %.

### 2.4 Reconocer entidades

Aplicamos NER con spaCy (`es_core_news_lg`) a las oraciones de los fragmentos contaminados de desarrollo y a la misma cantidad de oraciones de sinopsis limpias.

In [22]:
import spacy
nlp = spacy.load('es_core_news_lg')

oraciones_frag = [o for f in desarrollo[desarrollo['contaminado'] == 1]['fragmento'] for o in oraciones(f)]
oraciones_limpias = [o for s in desarrollo[desarrollo['contaminado'] == 0]['synopsis'] for o in oraciones(s)]
oraciones_limpias = pd.Series(oraciones_limpias).sample(len(oraciones_frag), random_state=42).tolist()

def contar_entidades(lista):
    etiquetas = Counter()
    textos = defaultdict(Counter)
    for doc in nlp.pipe(lista):
        for ent in doc.ents:
            etiquetas[ent.label_] += 1
            textos[ent.label_][ent.text] += 1
    return etiquetas, textos

ent_frag, textos_frag = contar_entidades(oraciones_frag)
ent_limp, textos_limp = contar_entidades(oraciones_limpias)

print(f"Oraciones de cada grupo: {len(oraciones_frag)}\n")
print(pd.DataFrame({'fragmentos': ent_frag, 'limpias': ent_limp}).fillna(0).astype(int))
for etiqueta in ['ORG', 'MISC', 'LOC', 'PER']:
    print(f"\n{etiqueta} en fragmentos: {textos_frag[etiqueta].most_common(8)}")
    print(f"{etiqueta} en limpias:    {textos_limp[etiqueta].most_common(8)}")

Oraciones de cada grupo: 88

      fragmentos  limpias
MISC          37       32
LOC           15       27
PER           16       49
ORG            2        5

ORG en fragmentos: [('LetraTotal', 1), ('com', 1)]
ORG en limpias:    [('Mendéleiev', 1), ('Marina', 1), ('Tercer Reich', 1), ('Dueño del Tiempo', 1), ('PLANTAR', 1)]

MISC en fragmentos: [('PDF', 6), ('Download this ebook for FREE', 4), ('sign-up required', 4), ('EPUB sin registro', 4), ('Descargalo', 4), ('EPUB', 2), ('LibroLibre', 2), ('EpubMax', 2)]
MISC en limpias:    [('Aprender', 3), ('¿Cómo', 1), ('¿Cómo vencer el miedo', 1), ('Fragmento', 1), ('La conciencia', 1), ('¿Patines cohete', 1), ('¿Quieres', 1), ('Mark', 1)]

LOC en fragmentos: [('Oferta', 2), ('Visitá pagin\u2060azona.invalid', 1), ('Baixe', 1), ('grátis agora mesmo', 1), ('ventas@paginatotal.example', 1), ('librоexpress.invalid', 1), ('Visitá rela\u200btohub.test', 1), ('Visitá tomo\u200bflash.test', 1)]
LOC en limpias:    [('Haruf', 2), ('Andes', 1), ('Gor',

**Resultados 2.4.** En 88 oraciones de cada grupo, las sinopsis limpias tienen muchas más personas (PER 49 contra 16) y lugares reales (Andes, España, la Tierra). En el texto insertado aparecen sobre todo entidades MISC con nombres de formatos y servicios: «PDF», «EPUB sin registro», «Download this ebook for FREE», «EpubMax», «LibroLibre». También aparecen nombres de marca como ORG («LetraTotal»). Además, muchas entidades del texto insertado son errores del modelo: etiqueta como persona los verbos en imperativo y voseo («compartila», «Ganá») y como lugar los dominios disimulados o las palabras en portugués («Oferta», «Baixe», «librоexpress.invalid»). Esto pasa porque `es_core_news_lg` está entrenado con texto periodístico y no conoce este registro publicitario.

**¿Sirven para detectar?** No por sí solas. Las sinopsis normales también tienen MISC y ORG (títulos, obras, organizaciones) y las etiquetas en el texto insertado son ruidosas, así que un detector basado en entidades marcaría muchos libros legítimos. Sirven para **explicar** lo que encontraron otros detectores: en un libro marcado por la regex o el clasificador, el NER señala la marca o el servicio que se promociona (EpubMax, LetraTotal) y ayuda a la revisión manual.

### 2.5 Clasificar oraciones

Armamos el conjunto de oraciones con los libros de desarrollo: las oraciones del fragmento son positivas y las de las sinopsis limpias son negativas. Las oraciones originales de los libros contaminados no se usan (no sabemos con certeza que sean limpias una vez insertado el texto). Una sinopsis recibe como puntaje la probabilidad más alta entre sus oraciones.

Para elegir el umbral sin mirar la prueba y sin usar probabilidades sobre oraciones que el modelo ya vio, calculamos los puntajes de desarrollo con validación cruzada por libro (5 particiones): cada libro se puntúa con un modelo entrenado sin él. Sobre esos puntajes aplicamos `precision_recall_curve` y tomamos el menor umbral que alcanza precisión 0.90.

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import StratifiedKFold
from sentence_transformers import SentenceTransformer

# Modelo de embeddings de oraciones multilingüe (el mismo que usamos en el TP2)
modelo_emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Guardamos los embeddings ya calculados para no repetirlos
cache_emb = {}

def embeber(textos):
    textos = list(textos)
    nuevos = list({t for t in textos if t not in cache_emb})
    if nuevos:
        vectores = modelo_emb.encode(nuevos, batch_size=64, show_progress_bar=len(nuevos) > 2000)
        cache_emb.update(zip(nuevos, vectores))
    return np.array([cache_emb[t] for t in textos])

def oraciones_entrenamiento(libros):
    textos, etiquetas = [], []
    for _, fila in libros.iterrows():
        if fila['contaminado'] == 1:
            nuevas = oraciones(fila['fragmento']) or [fila['fragmento']]
            textos += nuevas
            etiquetas += [1] * len(nuevas)
        else:
            nuevas = oraciones(fila['synopsis'])
            textos += nuevas
            etiquetas += [0] * len(nuevas)
    return textos, etiquetas

def entrenar_detector(textos, etiquetas, representacion):
    if representacion == 'tfidf':
        vec = TfidfVectorizer()
        X = vec.fit_transform(textos)
        transformar = vec.transform
    else:
        X = embeber(textos)
        transformar = embeber
    clf = LogisticRegression(class_weight='balanced', max_iter=1000)
    clf.fit(X, etiquetas)
    return lambda lista: clf.predict_proba(transformar(lista))[:, 1]

def puntaje_libros(predecir, sinopsis, min_palabras=0):
    # El puntaje de una sinopsis es la probabilidad máxima entre sus oraciones
    listas = [[o for o in oraciones(s) if len(o.split()) >= min_palabras] or [s] for s in sinopsis]
    probs = predecir([o for l in listas for o in l])
    puntajes, i = [], 0
    for l in listas:
        puntajes.append(probs[i:i + len(l)].max())
        i += len(l)
    return np.array(puntajes)

def puntajes_desarrollo(representacion, min_palabras=0):
    puntajes = np.zeros(len(desarrollo))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for idx_tr, idx_va in skf.split(desarrollo, desarrollo['contaminado']):
        t, e = oraciones_entrenamiento(desarrollo.iloc[idx_tr])
        predecir = entrenar_detector(t, e, representacion)
        puntajes[idx_va] = puntaje_libros(predecir, desarrollo['synopsis'].iloc[idx_va], min_palabras)
    return puntajes

def elegir_umbral(y, puntajes, precision_objetivo=0.90):
    p, r, u = precision_recall_curve(y, puntajes)
    i = np.where(p[:-1] >= precision_objetivo)[0][0]
    return u[i], p[i], r[i]

textos_dev, etiquetas_dev = oraciones_entrenamiento(desarrollo)
print(f"Oraciones de entrenamiento: {len(textos_dev)} (positivas: {sum(etiquetas_dev)})")

Oraciones de entrenamiento: 1831 (positivas: 88)


In [24]:
detectores_clf = {}
umbrales = {}
puntajes_dev = {}
for rep in ['tfidf', 'embeddings']:
    puntajes = puntajes_desarrollo(rep)
    puntajes_dev[rep] = puntajes
    umbral, p, r = elegir_umbral(desarrollo['contaminado'], puntajes)
    umbrales[rep] = umbral
    print(f"{rep}: umbral {umbral:.3f} -> desarrollo: precisión {p:.2f}, recall {r:.2f}")
    print(f"   con umbral 0.5 marcaría {(puntajes >= 0.5).sum()} libros de desarrollo; con el elegido, {(puntajes >= umbral).sum()}")
    # Modelo final: entrenado con todas las oraciones de desarrollo
    detectores_clf[rep] = entrenar_detector(textos_dev, etiquetas_dev, rep)

puntaje_prueba = {rep: puntaje_libros(detectores_clf[rep], prueba['synopsis']) for rep in detectores_clf}
medir("clasificador TF-IDF", puntaje_prueba['tfidf'] >= umbrales['tfidf'], prueba)
medir("clasificador embeddings", puntaje_prueba['embeddings'] >= umbrales['embeddings'], prueba)

tfidf: umbral 0.380 -> desarrollo: precisión 0.90, recall 0.93
   con umbral 0.5 marcaría 59 libros de desarrollo; con el elegido, 72


embeddings: umbral 0.714 -> desarrollo: precisión 0.91, recall 0.83
   con umbral 0.5 marcaría 74 libros de desarrollo; con el elegido, 64


clasificador TF-IDF: precisión 0.94, recall 0.97 (31 libros marcados)
clasificador embeddings: precisión 0.86, recall 0.83 (29 libros marcados)


Para ver en qué tipo de contaminación gana cada representación, asignamos a cada fragmento de prueba un tipo según su forma: enlace visible (la regex lo encuentra sin normalizar), enlace disimulado (solo después de normalizar), otro idioma, o texto promocional sin enlaces.

In [25]:
def tipo_fragmento(fragmento):
    if detector_regex(fragmento):
        return 'enlace visible'
    if detector_regex_norm(fragmento):
        return 'enlace disimulado'
    if idioma(fragmento) != 'es':
        return 'otro idioma'
    return 'promocional'

prueba = prueba.copy()
prueba['tipo'] = np.where(prueba['contaminado'] == 1, prueba['fragmento'].apply(tipo_fragmento), 'limpio')
prueba['clf_tfidf'] = puntaje_prueba['tfidf'] >= umbrales['tfidf']
prueba['clf_emb'] = puntaje_prueba['embeddings'] >= umbrales['embeddings']

contaminados_prueba = prueba[prueba['contaminado'] == 1]
print(contaminados_prueba.groupby('tipo')[['clf_tfidf', 'clf_emb']].agg(['sum', 'count']))

                  clf_tfidf       clf_emb      
                        sum count     sum count
tipo                                           
enlace disimulado         3     4       2     4
enlace visible           15    15      12    15
otro idioma               5     5       5     5
promocional               6     6       6     6


In [26]:
# Generalización a frases nuevas: probamos mensajes promocionales que no están en la muestra
frases_nuevas = [
    "Conseguí gratis la colección completa del autor, solo por tiempo limitado.",
    "Unite a nuestro club de lectores y recibí descuentos exclusivos todas las semanas.",
    "Bajá ya mismo la novela entera desde nuestra aplicación, sin costo.",
    "Free download of the complete novel, only for this week, don't miss it!",
    "Una novela que cambió para siempre la historia de la literatura del siglo XX.",
]
for f in frases_nuevas:
    p_t = detectores_clf['tfidf']([f])[0]
    p_e = detectores_clf['embeddings']([f])[0]
    print(f"TF-IDF {p_t:.2f} (umbral {umbrales['tfidf']:.2f}) | embeddings {p_e:.2f} (umbral {umbrales['embeddings']:.2f}) | {f}")

TF-IDF 0.86 (umbral 0.38) | embeddings 1.00 (umbral 0.71) | Conseguí gratis la colección completa del autor, solo por tiempo limitado.
TF-IDF 0.22 (umbral 0.38) | embeddings 0.96 (umbral 0.71) | Unite a nuestro club de lectores y recibí descuentos exclusivos todas las semanas.
TF-IDF 0.29 (umbral 0.38) | embeddings 0.80 (umbral 0.71) | Bajá ya mismo la novela entera desde nuestra aplicación, sin costo.
TF-IDF 0.55 (umbral 0.38) | embeddings 1.00 (umbral 0.71) | Free download of the complete novel, only for this week, don't miss it!
TF-IDF 0.04 (umbral 0.38) | embeddings 0.00 (umbral 0.71) | Una novela que cambió para siempre la historia de la literatura del siglo XX.


**Resultados 2.5.** Con umbral 0.5 el modelo TF-IDF marcaría 59 libros de desarrollo y el de embeddings 74: las probabilidades no son comparables. Elegimos un umbral por modelo para lograr precisión 0.90 en desarrollo: **0.38 para TF-IDF** y **0.71 para embeddings**. En prueba, a esa precisión:

| | precisión | recall |
|---|---|---|
| TF-IDF | 0.94 | 0.97 |
| Embeddings | 0.86 | 0.83 |

**¿Por qué clasificar oraciones y no la sinopsis completa?** El texto insertado es una o dos oraciones dentro de una sinopsis de 8 a 15. Si se representa la sinopsis entera, el vector queda dominado por el contenido del libro y la señal del spam se diluye (lo mismo que pasó con la detección de idioma). Además, las etiquetas de oración son más limpias: sabemos exactamente qué texto es el insertado (la columna `fragmento`). Por último, el puntaje «oración más sospechosa» indica qué oración causó la marca.

**¿Qué representación detecta mejor los contaminados de prueba?** TF-IDF, en todos los tipos. La mayor diferencia está en los enlaces: TF-IDF encuentra el 100 % de los enlaces visibles y el 75 % de los disimulados; embeddings, el 80 % y el 50 %. Los dominios inventados (`tomozona.example`, `lectorya.test`) son tokens con señal directa para TF-IDF. Para el modelo de embeddings, en cambio, una URL fragmentada en subpalabras no tiene un significado que la distinga. En texto promocional y en otro idioma los dos detectan el 100 %.

**¿Cuál generaliza mejor a mensajes nuevos?** Embeddings. Probamos frases promocionales que no están en la muestra: «Unite a nuestro club de lectores y recibí descuentos exclusivos...» recibe 0.96 con embeddings (marcada) y 0.22 con TF-IDF (no marcada). «Bajá ya mismo la novela entera desde nuestra aplicación, sin costo» recibe 0.80 y 0.29. TF-IDF solo reconoce las palabras exactas que vio («gratis», «oferta», «descargá»). Los embeddings reconocen el **sentido** promocional aunque cambien las palabras, y también en otro idioma, porque el modelo es multilingüe. La frase legítima de control queda baja en los dos. Conclusión: TF-IDF es mejor para cadenas literales como URLs y para las variantes que ya están en la muestra. Embeddings es mejor para mensajes nuevos con el mismo sentido. Como el catálogo tiene mensajes que no aparecen en la muestra, conviene combinar los dos.

### 2.6 Encontrar libros copiados

Buscamos para cada libro del catálogo su vecino más parecido con `NearestNeighbors(metric="cosine")` sobre TF-IDF. Revisamos los pares con similitud alta y sin autores en común. Para cada par calculamos la similitud de Jaccard sobre conjuntos de palabras y la proporción de palabras del libro más corto que aparecen en el más largo. Como la copia es el texto original más un agregado, la copia es el libro con la sinopsis más larga.

In [27]:
from sklearn.neighbors import NearestNeighbors

catalogo = df_final.reset_index(drop=True)
vec_cat = TfidfVectorizer()
X_cat = vec_cat.fit_transform(catalogo['synopsis'])
nn = NearestNeighbors(n_neighbors=2, metric='cosine').fit(X_cat)
distancias, indices = nn.kneighbors(X_cat)

def palabras(texto):
    return set(re.findall(r"\w+", texto.lower()))

def jaccard(a, b):
    return len(a & b) / len(a | b)

pares = []
vistos = set()
for i in range(len(catalogo)):
    j = indices[i, 1]
    sim = 1 - distancias[i, 1]
    clave = tuple(sorted((i, j)))
    if clave in vistos:
        continue
    vistos.add(clave)
    if set(catalogo.loc[i, 'authors']) & set(catalogo.loc[j, 'authors']):
        continue
    corto, largo = sorted((i, j), key=lambda k: len(catalogo.loc[k, 'synopsis']))
    p_corto, p_largo = palabras(catalogo.loc[corto, 'synopsis']), palabras(catalogo.loc[largo, 'synopsis'])
    pares.append({
        'similitud': round(sim, 3),
        'jaccard': round(jaccard(p_corto, p_largo), 3),
        'contencion': round(len(p_corto & p_largo) / len(p_corto), 3),
        'copia': catalogo.loc[largo, 'slug'], 'autor_copia': catalogo.loc[largo, 'authors_str'],
        'original': catalogo.loc[corto, 'slug'], 'autor_original': catalogo.loc[corto, 'authors_str'],
    })
pares = pd.DataFrame(pares).sort_values('similitud', ascending=False)

# Cantidad de pares con autores distintos según el umbral de similitud
for u in [0.95, 0.9, 0.85, 0.8, 0.7, 0.6, 0.5]:
    print(f"similitud >= {u}: {(pares['similitud'] >= u).sum()} pares")

similitud >= 0.95: 29 pares
similitud >= 0.9: 56 pares
similitud >= 0.85: 64 pares
similitud >= 0.8: 65 pares
similitud >= 0.7: 75 pares
similitud >= 0.6: 77 pares
similitud >= 0.5: 92 pares


In [28]:
# Revisamos la zona donde se corta la cantidad de pares
display(pares[(pares['similitud'] >= 0.6)].head(10))
display(pares[(pares['similitud'] >= 0.6) & (pares['similitud'] < 0.9)])

,similitud,jaccard,contencion,copia,autor_copia,original,autor_original
1039,1.000,1.000,1.0,las-grandes-preguntas-los-ninos-las-sencillas-...,Gemma Elwin Harris,las-grandes-preguntas-de-los-ninos-y-las-senci...,Varios Autores
3948,0.988,0.954,1.0,los-hijos-de-la-promesa,Paula Arrieta,elvira-de-hidalgo,Juan Villalba Sebastián
1619,0.988,0.944,1.0,la-ultima-tormenta,Lucía Oyarzún,la-pequena-comunista-no-sonreia-nunca,Lola Lafon
4935,0.985,0.954,1.0,bajo-la-isla,Clara Correa,tras-os-montes,Julio Llamazares
220,0.984,0.926,1.0,richard-p-feynman-la-fisica-de-las-palabras,Michelle Feynman,la-fisica-de-las-palabras,Richard Phillips Feynman
320,0.977,0.922,1.0,cronica-del-bosque,Sofía Correa,cotufas-en-el-golfo,Gonzalo Torrente Ballester
4247,0.977,0.936,1.0,la-ultima-herencia,Andrés Barrionuevo,en-frio,Bjørn Lomborg
4441,0.973,0.918,1.0,bajo-la-colmena,Andrés Oyarzún,teleshakespeare,Jorge Carrión
3817,0.973,0.928,1.0,el-secreto-del-desierto,Ignacio Arrieta,la-retirada,Noam Chomsky Vijay Prashad
391,0.971,0.908,1.0,memorias-del-bosque-4,Valeria Ledesma,los-conquistadores-de-lo-inutil,Lionel Terray


,similitud,jaccard,contencion,copia,autor_copia,original,autor_original
4338,0.897,0.766,1.000,el-camino-de-la-noche,Valeria Arrieta,sin-novedad-en-el-frente-erich-maria-remarque,Erich Maria Remarque
1046,0.892,0.747,1.000,sombras-sobre-el-bosque,Paula Oyarzún,bodas-de-sangre,Federico García Lorca
1882,0.888,0.797,1.000,cronica-del-relojero,Sofía Correa,enciclopedia-eslava,Juan Eslava Galán
851,0.887,0.758,1.000,memorias-del-relojero,Tomás Oyarzún,presidio,Randy Kennedy
3903,0.866,0.722,1.000,sombras-sobre-el-relojero,Julián Ledesma,misterio-del-fugitivo,Enid Blyton
4238,0.861,0.735,1.000,los-hijos-de-la-frontera,Ignacio Vidal,cartas-marruecas,José de Cadalso
2388,0.859,0.738,1.000,memorias-del-bosque-2,Clara Arrieta,como-crear-una-mente,Ray Kurzweil
2686,0.850,0.760,1.000,los-hijos-de-la-colmena,Paula Peralta,historia-de-la-filosofia,Frederick Copleston
2727,0.807,0.717,1.000,los-hijos-de-la-llanura,Lucía Correa,la-fascinante-historia-de-las-palabras,Ricardo Soca
836,0.796,0.662,0.828,quien-secuestro-a-la-princesa-saralinda,Bill Stine Megan Stine,la-venganza-de-la-reina-de-dragones,Eric Affabee


In [29]:
UMBRAL_COPIA = 0.75
CONTENCION_MINIMA = 0.95

copias = pares[(pares['similitud'] >= UMBRAL_COPIA) & (pares['contencion'] >= CONTENCION_MINIMA)]
slugs_copia = set(copias['copia'])
print(f"Libros marcados como copia: {len(slugs_copia)}")

medir("libro copiado", prueba['slug'].isin(slugs_copia), prueba)

Libros marcados como copia: 66
libro copiado: precisión 1.00, recall 0.10 (3 libros marcados)


**Resultados 2.6.** Hay un corte claro. Los pares sin autores en común con contención 1.0 (todas las palabras del libro corto están en el largo) son copias hasta una similitud de 0.78. Ejemplos: *cronica-del-bosque* (Sofía Correa) copia *cotufas-en-el-golfo* de Torrente Ballester, y *sombras-sobre-el-rio-2* (Andrés Barrionuevo) copia *Peter Pan*. Los libros copiados comparten un patrón: autores inventados con nombres que se repiten (Vidal, Arrieta, Correa, Oyarzún, Peralta, Barrionuevo, Ledesma) y títulos genéricos («Memorias del bosque», «Los hijos de la llanura», «Crónica del faro»), varias veces con sufijo `-2`, `-3`, `-4` en el slug.

**¿Cuál es la copia?** La que tiene la sinopsis más larga: copia el texto completo del original y le agrega el mensaje (por eso la contención es 1.0 y el Jaccard es menor que 1). El original es el libro real y no se elimina.

Con similitudes entre 0.72 y 0.80 aparecen **series legítimas**: los libros de la colección «Elige tu propia aventura» (R. L. Stine, Lynn Beach, Scott Siegel...) comparten el texto de presentación de la colección, pero cada uno tiene su propio argumento. Ahí la contención baja a 0.73–0.83. Por eso el criterio combina **similitud ≥ 0.75 y contención ≥ 0.95**. Entre los marcados quedan pocos casos legítimos que el criterio no puede separar: dos ediciones del mismo libro cargadas con autores escritos distinto (por ejemplo, el libro de Feynman cargado a nombre de *Michelle Feynman*). Son otra versión de un libro que ya está, así que eliminarlos no saca una obra del catálogo.

En prueba la precisión es 1.00 y el recall 0.10: solo 3 de los 30 contaminados de prueba son libros inventados. El detector es muy específico. Además, estos libros también tienen el texto agregado, que los otros detectores pueden encontrar.

### 2.7 Combinar y entregar

In [30]:
prueba['regex'] = prueba['synopsis'].apply(detector_regex)
prueba['regex_norm'] = prueba['synopsis'].apply(detector_regex_norm)
prueba['idioma'] = prueba['synopsis'].apply(detector_idioma)
prueba['copia'] = prueba['slug'].isin(slugs_copia)

# Primero combinamos los cinco detectores
TODOS = ['regex_norm', 'idioma', 'clf_tfidf', 'clf_emb', 'copia']
medir("combinación de los cinco", prueba[TODOS].any(axis=1), prueba)

# El detector de idioma no aporta recall que no den los clasificadores y suma falsos positivos: lo dejamos afuera
DETECTORES = ['regex_norm', 'clf_tfidf', 'clf_emb', 'copia']
prueba['combinacion'] = prueba[DETECTORES].any(axis=1)
medir("combinación sin idioma", prueba['combinacion'], prueba)

tabla = pd.DataFrame(resultados).T
display(tabla)

print("Recall por tipo de contaminación en prueba:")
display(prueba[prueba['contaminado'] == 1].groupby('tipo')[['regex', 'regex_norm', 'idioma', 'clf_tfidf', 'clf_emb', 'copia', 'combinacion']].mean().round(2))

combinación de los cinco: precisión 0.86, recall 1.00 (35 libros marcados)
combinación sin idioma: precisión 0.88, recall 1.00 (34 libros marcados)


,precision,recall,marcados
regex mínima,1.00,0.13,4.0
regex,1.00,0.50,15.0
regex + normalización,1.00,0.63,19.0
idioma por oración,0.89,0.27,9.0
clasificador TF-IDF,0.94,0.97,31.0
clasificador embeddings,0.86,0.83,29.0
libro copiado,1.00,0.10,3.0
combinación de los cinco,0.86,1.00,35.0
combinación sin idioma,0.88,1.00,34.0


Recall por tipo de contaminación en prueba:


,regex,regex_norm,idioma,clf_tfidf,clf_emb,copia,combinacion
tipo,,,,,,,
enlace disimulado,0.0,1.0,0.0,0.75,0.5,0.00,1.0
enlace visible,1.0,1.0,0.2,1.00,0.8,0.13,1.0
otro idioma,0.0,0.0,1.0,1.00,1.0,0.00,1.0
promocional,0.0,0.0,0.0,1.00,1.0,0.17,1.0


In [31]:
# Falsos positivos en prueba: qué detector los marcó
fp = prueba[prueba[TODOS].any(axis=1) & (prueba['contaminado'] == 0)]
for _, fila in fp.iterrows():
    print(fila['slug'], '->', [d for d in TODOS if fila[d]])

el-haiku-japones -> ['clf_emb']
el-rey-de-amarillo -> ['idioma']
agentes-de-la-noche -> ['clf_tfidf', 'clf_emb']
el-disco-de-newton -> ['clf_emb']
cuentos-completos-penguin-clasicos -> ['clf_tfidf', 'clf_emb']


Aplicamos estos detectores al catálogo completo. Los clasificadores tienen que recorrer todas las oraciones, por eso esta celda tarda unos minutos. Guardamos el puntaje de cada libro y la oración que lo produjo, para revisar qué se marca.

In [32]:
def oracion_mas_sospechosa(predecir, texto, min_palabras=0):
    partes = [o for o in oraciones(texto) if len(o.split()) >= min_palabras] or [texto]
    probs = predecir(partes)
    return partes[probs.argmax()]

auditoria = catalogo[['slug', 'title', 'authors_str', 'synopsis']].copy()
auditoria['regex_norm'] = auditoria['synopsis'].apply(detector_regex_norm)
auditoria['copia'] = auditoria['slug'].isin(slugs_copia)
for rep, col in [('tfidf', 'clf_tfidf'), ('embeddings', 'clf_emb')]:
    auditoria[f'p_{rep}'] = puntaje_libros(detectores_clf[rep], auditoria['synopsis'])
    auditoria[col] = auditoria[f'p_{rep}'] >= umbrales[rep]

print(auditoria[DETECTORES].sum())
print(f"Marcados por al menos uno: {auditoria[DETECTORES].any(axis=1).sum()} de {len(auditoria)}")

regex_norm    180
clf_tfidf     399
clf_emb       433
copia          66
dtype: int64
Marcados por al menos uno: 625 de 8575


In [33]:
# Revisamos a mano los libros que marcó un solo clasificador y ningún otro detector
solo_tfidf = auditoria['clf_tfidf'] & ~auditoria[['regex_norm', 'clf_emb', 'copia']].any(axis=1)
solo_emb = auditoria['clf_emb'] & ~auditoria[['regex_norm', 'clf_tfidf', 'copia']].any(axis=1)
print(f"Solo TF-IDF: {solo_tfidf.sum()} | Solo embeddings: {solo_emb.sum()}")

for nombre, mascara, rep in [('solo TF-IDF', solo_tfidf, 'tfidf'), ('solo embeddings', solo_emb, 'embeddings')]:
    print(f"\n=== {nombre}: 20 al azar, con la oración de mayor puntaje ===")
    for _, fila in auditoria[mascara].sample(20, random_state=0).iterrows():
        o = oracion_mas_sospechosa(detectores_clf[rep], fila['synopsis'])
        print(f"  {fila[f'p_{rep}']:.2f} ({len(o.split())} palabras) | {o[:120]}")

Solo TF-IDF: 125 | Solo embeddings: 186

=== solo TF-IDF: 20 al azar, con la oración de mayor puntaje ===
  0.43 (1 palabras) | Imprescindible.
  0.41 (15 palabras) | Así, podrás recargar tus baterías y tu relación amorosa no caerá en la rutina .
  0.42 (16 palabras) | Sean cuales fueren tus elecciones, tendrás aventura y acción en los libros de «Aventura sin fin».
  0.51 (37 palabras) | Aunque Cradag, tu padre, desea que te conviertas en guerrero, tus inclinaciones te llevan a imitar a tu madre y aprender
  0.52 (4 palabras) | Ahora es tu turno.
  0.60 (4 palabras) | ¿Malgastas tu energía enfadándote?
  0.55 (40 palabras) | Actualmente existen muchas herramientas, como el Bullet Journal o el método Pomodoro , que te ayudarán a aumentar tu pro
  0.43 (4 palabras) | ¿Cuál es tu tipo?
  0.45 (35 palabras) | De este modo, el lector verá desfilar desde Loïe Fuller hasta Zara Whites, desde Duelo al sol hasta Tras la puerta verde
  0.56 (37 palabras) | Cuando hablamos de víctimas de violació

**Lo que muestra la revisión.** Combinados con «al menos uno», los detectores marcan 625 libros del catálogo (7.3 %). Muchos los marca un solo clasificador:

- **Solo TF-IDF (125 libros):** casi todos son falsos positivos. Son sinopsis legítimas de autoayuda, libros juveniles o de «elige tu aventura» escritas en segunda persona («tu camino está trazado», «te acompaña», «tus elecciones»). En los fragmentos de entrenamiento abundan las formas «te», «tu», «gratis», «libro», así que el modelo aprendió palabras sueltas. Sus puntajes quedan apenas por encima del umbral (0.38–0.60). La excepción son los pocos casos con puntaje alto, que sí son spam («...Si te gustó esta reseña, compartila...»).
- **Solo embeddings (186 libros):** aparecen **mensajes nuevos que no están en la muestra** y que los embeddings reconocen por su sentido: «Promoción exclusiva para nuevos usuarios: pagás uno y te llevás la saga entera», «Join millions of readers and claim your free premium membership...», «Inscribite ahora y obtené acceso ilimitado a audiolibros...». Es justamente la generalización que esperábamos en 2.5. Pero también hay falsos positivos, sobre todo en **oraciones muy cortas** («Chicago Tribune», «Varios reportajes.», «Vivimos en línea.»): con pocas palabras el embedding no tiene contexto, lo mismo que pasaba con la detección de idioma.
- En la misma revisión apareció una variante de enlace disimulado que no estaba en la muestra: «capitulototal **(punto)** invalid». La agregamos a `normalizar`.

**Ajuste de la combinación** (los umbrales se eligen sobre desarrollo, no sobre el catálogo):

1. Los clasificadores ignoran las oraciones de menos de **6 palabras**. El fragmento contaminado más corto de la muestra tiene 6.
2. Un libro queda marcado por los clasificadores si **los dos coinciden** (cada uno con su umbral de 2.5), **o** si **uno solo** supera un umbral de **alta confianza**: el que da precisión 0.98 en desarrollo con validación cruzada. Así conservamos la capacidad de los embeddings para encontrar mensajes nuevos y descartamos las marcas débiles de un solo modelo.

In [34]:
MIN_PALABRAS_CLF = 6

# Umbrales de alta confianza: puntajes de desarrollo (validación cruzada) ignorando las oraciones cortas
umbral_alto = {}
for rep in ['tfidf', 'embeddings']:
    puntajes = puntajes_desarrollo(rep, MIN_PALABRAS_CLF)
    umbral_alto[rep], p, r = elegir_umbral(desarrollo['contaminado'], puntajes, precision_objetivo=0.98)
    print(f"Umbral de alta confianza {rep}: {umbral_alto[rep]:.3f} -> desarrollo: precisión {p:.2f}, recall {r:.2f}")

def clasificadores_final(p_tfidf, p_emb):
    acuerdo = (p_tfidf >= umbrales['tfidf']) & (p_emb >= umbrales['embeddings'])
    return acuerdo | (p_tfidf >= umbral_alto['tfidf']) | (p_emb >= umbral_alto['embeddings'])

p6_tfidf = puntaje_libros(detectores_clf['tfidf'], prueba['synopsis'], MIN_PALABRAS_CLF)
p6_emb = puntaje_libros(detectores_clf['embeddings'], prueba['synopsis'], MIN_PALABRAS_CLF)
prueba['clasificadores_final'] = clasificadores_final(p6_tfidf, p6_emb)
prueba['final'] = prueba['regex_norm'] | prueba['copia'] | prueba['clasificadores_final']

medir("clasificadores (acuerdo o uno con alta confianza)", prueba['clasificadores_final'], prueba)
medir("COMBINACIÓN FINAL", prueba['final'], prueba)

tabla = pd.DataFrame(resultados).T
display(tabla)
print("Recall por tipo de contaminación en prueba:")
display(prueba[prueba['contaminado'] == 1].groupby('tipo')[['regex_norm', 'clasificadores_final', 'copia', 'final']].mean().round(2))

Umbral de alta confianza tfidf: 0.697 -> desarrollo: precisión 0.98, recall 0.71


Umbral de alta confianza embeddings: 0.786 -> desarrollo: precisión 0.98, recall 0.80
clasificadores (acuerdo o uno con alta confianza): precisión 0.93, recall 0.90 (29 libros marcados)
COMBINACIÓN FINAL: precisión 0.94, recall 1.00 (32 libros marcados)


,precision,recall,marcados
regex mínima,1.00,0.13,4.0
regex,1.00,0.50,15.0
regex + normalización,1.00,0.63,19.0
idioma por oración,0.89,0.27,9.0
clasificador TF-IDF,0.94,0.97,31.0
clasificador embeddings,0.86,0.83,29.0
libro copiado,1.00,0.10,3.0
combinación de los cinco,0.86,1.00,35.0
combinación sin idioma,0.88,1.00,34.0
clasificadores (acuerdo o uno con alta confianza),0.93,0.90,29.0


Recall por tipo de contaminación en prueba:


,regex_norm,clasificadores_final,copia,final
tipo,,,,
enlace disimulado,1.0,0.75,0.00,1.0
enlace visible,1.0,0.87,0.13,1.0
otro idioma,0.0,1.00,0.00,1.0
promocional,0.0,1.00,0.17,1.0


Aplicamos la combinación final al catálogo completo y guardamos los archivos. Para los libros de la muestra etiquetada usamos la etiqueta conocida.

In [35]:
a6_tfidf = puntaje_libros(detectores_clf['tfidf'], auditoria['synopsis'], MIN_PALABRAS_CLF)
a6_emb = puntaje_libros(detectores_clf['embeddings'], auditoria['synopsis'], MIN_PALABRAS_CLF)
auditoria['acuerdo'] = (a6_tfidf >= umbrales['tfidf']) & (a6_emb >= umbrales['embeddings'])
auditoria['tfidf_alto'] = a6_tfidf >= umbral_alto['tfidf']
auditoria['emb_alto'] = a6_emb >= umbral_alto['embeddings']

NOMBRES_MOTIVO = {'regex_norm': 'regex', 'copia': 'copia', 'acuerdo': 'clasificadores',
                  'tfidf_alto': 'tfidf_alto', 'emb_alto': 'embeddings_alto'}
auditoria['motivo'] = auditoria[list(NOMBRES_MOTIVO)].apply(
    lambda fila: '+'.join(m for d, m in NOMBRES_MOTIVO.items() if fila[d]), axis=1)

etiqueta = dict(zip(muestra['slug'], muestra['contaminado']))
auditoria['etiqueta'] = auditoria['slug'].map(etiqueta)
auditoria.loc[auditoria['etiqueta'] == 0, 'motivo'] = ''
auditoria.loc[(auditoria['etiqueta'] == 1) & (auditoria['motivo'] == ''), 'motivo'] = 'muestra_etiquetada'

sospechosos = auditoria[auditoria['motivo'] != ''][['slug', 'motivo']]
GRUPO = 'grupo_marquardt_aguirre_venesia_zappala'
sospechosos.to_csv('sospechosos.csv', index=False)
sospechosos.to_csv(f'{GRUPO}_sospechosos.csv', index=False)

catalogo_limpio = catalogo[~catalogo['slug'].isin(sospechosos['slug'])][['slug', 'title', 'authors_str', 'synopsis']]
catalogo_limpio.to_csv('catalogo_limpio.csv', index=False)

print(f"Libros sospechosos: {len(sospechosos)} ({len(sospechosos) / len(catalogo):.1%} del catálogo)")
print(sospechosos['motivo'].value_counts())
print(f"\nCatálogo limpio: {len(catalogo_limpio)} libros")

Libros sospechosos: 423 (4.9% del catálogo)
motivo
embeddings_alto                                          121
clasificadores+tfidf_alto+embeddings_alto                 73
regex+clasificadores+tfidf_alto+embeddings_alto           69
regex                                                     52
regex+copia+clasificadores+tfidf_alto+embeddings_alto     30
copia+clasificadores+tfidf_alto+embeddings_alto           22
clasificadores+embeddings_alto                            18
regex+clasificadores+embeddings_alto                       8
regex+tfidf_alto                                           7
regex+copia                                                6
regex+embeddings_alto                                      3
tfidf_alto                                                 3
clasificadores                                             2
copia+embeddings_alto                                      2
regex+copia+tfidf_alto                                     2
regex+copia+clasificadores+embeddi

In [36]:
# Revisamos los libros marcados por un solo clasificador con alta confianza
for motivo, rep in [('embeddings_alto', 'embeddings'), ('tfidf_alto', 'tfidf')]:
    marcados = auditoria[auditoria['motivo'] == motivo]
    print(f"\n=== Libros marcados solo por {motivo}: {len(marcados)} (hasta 15 al azar) ===")
    for _, fila in marcados.sample(min(15, len(marcados)), random_state=0).iterrows():
        print("  -", oracion_mas_sospechosa(detectores_clf[rep], fila['synopsis'], MIN_PALABRAS_CLF)[:120])


=== Libros marcados solo por embeddings_alto: 121 (hasta 15 al azar) ===
  - Join millions of readers and claim your free premium membership before it expires.
  - Sumate a la comunidad LibroMax y recibí cada semana un libro elegido para vos, sin costo.
  - Join millions of readers and claim your free premium membership before it expires.
  - Nueve capítulos cortos para disfrutar de una entrada en la realidad.
  - Esta edición se basa en la edición digital de SADRAC.
  - Un libro útil, claro y accesible para todo aquel que busque emular los métodos del inversor más reconocido del mundo.
  - Assine hoje e ganhe três livros digitais de presente, sem nenhum custo adicional.
  - Promoción exclusiva para nuevos usuarios: pagás uno y te llevás la saga entera.
  - Esta guía te ofrece todos los trucos para que tu búsqueda sea un éxito, pero también destapa las trampas que debes evita
  - Aprovechá el descuento del 70 % solo por hoy; después vuelve a su precio normal.
  - Promoción exclusiva p

**Resultados 2.7.** Tabla final sobre la parte de prueba de la muestra (150 libros, 30 contaminados):

| Detector | Precisión | Recall |
|---|---|---|
| Regex mínima (enunciado) | 1.00 | 0.13 |
| Regex (URL, correo, teléfono) | 1.00 | 0.50 |
| Regex + normalización | 1.00 | 0.63 |
| Idioma por oración | 0.89 | 0.27 |
| Clasificador TF-IDF | 0.94 | 0.97 |
| Clasificador embeddings | 0.86 | 0.83 |
| Libro copiado | 1.00 | 0.10 |
| Combinación de los cinco (al menos uno) | 0.86 | 1.00 |
| Combinación sin idioma | 0.88 | 1.00 |
| Clasificadores: acuerdo o uno con alta confianza | 0.93 | 0.90 |
| **Combinación final** | **0.94** | **1.00** |

La combinación final es: regex con normalización, **o** libro copiado, **o** los dos clasificadores de acuerdo, **o** un clasificador solo con alta confianza. Detecta los 30 contaminados de prueba, en los cuatro tipos, con 2 falsos positivos. El detector de idioma quedó afuera de la combinación: todo lo que encuentra ya lo encuentran los clasificadores, que fueron entrenados con los mensajes en inglés y portugués, y solo agrega falsos positivos.

En el catálogo completo marcamos **423 libros (4.9 %)** y el catálogo limpio queda con **8152 libros**. La precisión real en el catálogo va a ser **menor** que la medida en la muestra. La muestra tiene 20 % de contaminados y el catálogo bastante menos, así que la misma tasa de falsos positivos sobre libros limpios pesa más. La revisión manual de los marcados solo por embeddings mostró 4 errores entre 15 casos revisados (oraciones como «Un libro útil, claro y accesible...» o «Esta edición se basa en la edición digital de SADRAC»).

**Costo de un falso positivo:** cada libro legítimo marcado se elimina del catálogo y nunca se va a poder recomendar. Por eso no usamos la combinación «al menos uno» de todos los detectores: en el catálogo completo marcaba 625 libros, y la revisión mostró que los marcados solo por TF-IDF eran casi todos sinopsis legítimas.

## Paso 3 Registrar las preferencias de un lector

Lector de referencia: **Agustín**.

Mezclamos el catálogo limpio una sola vez y exportamos los primeros libros de ese orden en una planilla. El lector completa la columna interes con 1 (me interesaría leerlo), 0 (no me interesaría) o vacío (duda o contaminado). Se registraron 111 respuestas válidas (56 de interés positivo y 55 de desinterés), manteniendo un balance del 50 % entre clases.

In [37]:
catalogo_limpio = pd.read_csv('catalogo_limpio.csv')
orden = catalogo_limpio.sample(frac=1, random_state=42).copy()

plantilla = orden[['slug', 'title', 'synopsis']].head(150).copy()
plantilla['interes'] = ''
plantilla.to_csv('preferencias_a_completar.csv', index=False)
print(f"Planilla para el lector: {len(plantilla)} libros")
display(plantilla[['slug', 'title']].head())

Planilla para el lector: 150 libros


,slug,title
5814,wittgenstein-la-consciencia-del-limite,Wittgenstein. La consciencia del límite
483,me-gusta-el-futbol,Me gusta el fútbol
7606,el-dia-d-la-batalla-de-normandia,El día D. La batalla de Normandía
3939,el-caso-del-tutor-en-apuros,El caso del tutor en apuros
3821,el-profesor-john-katzenbach,El profesor


Una vez que el lector completó la planilla, la guardamos como `preferencias.csv` y la subimos al entorno.

In [38]:
preferencias = pd.read_csv('preferencias.csv')

# El lector tiene que haber seguido el orden desde el principio
assert preferencias['slug'].tolist() == orden['slug'].head(len(preferencias)).tolist(), "preferencias.csv no sigue el orden de df.sample(frac=1, random_state=42)"

# Libros contaminados que el lector encontró y el paso 2 no detectó (completar con los slugs)
revision_manual = []
assert preferencias.loc[preferencias['slug'].isin(revision_manual), 'interes'].isna().all()

if revision_manual:
    sospechosos = pd.concat([sospechosos, pd.DataFrame({'slug': revision_manual, 'motivo': 'revision_manual'})])
    sospechosos.to_csv('sospechosos.csv', index=False)
    sospechosos.to_csv(f'{GRUPO}_sospechosos.csv', index=False)

validas = preferencias.dropna(subset=['interes']).copy()
validas['interes'] = validas['interes'].astype(int)
print(f"Libros recorridos: {len(preferencias)}")
print(f"Abstenciones (interes vacío): {preferencias['interes'].isna().sum()}")
print(f"Respuestas válidas: {len(validas)}")
print(validas['interes'].value_counts().rename(index={1: 'interes = 1', 0: 'interes = 0'}))

assert len(validas) >= 100 and (validas['interes'].value_counts() >= 20).all(), "Faltan respuestas: seguir valorando con el mismo orden"

Libros recorridos: 150
Abstenciones (interes vacío): 39
Respuestas válidas: 111
interes
interes = 1    56
interes = 0    55
Name: count, dtype: int64


## Paso 4 Separar los libros para aprender y evaluar

In [39]:
pref_train, pref_test = train_test_split(
    validas,
    test_size=0.2,
    random_state=42,
    stratify=validas['interes']
)

preferencias['conjunto'] = ''
preferencias.loc[preferencias['slug'].isin(pref_train['slug']), 'conjunto'] = 'entrenamiento'
preferencias.loc[preferencias['slug'].isin(pref_test['slug']), 'conjunto'] = 'prueba'
preferencias.to_csv('preferencias.csv', index=False)
preferencias.to_csv(f'{GRUPO}_preferencias.csv', index=False)

y_train = pref_train['interes'].values
y_test = pref_test['interes'].values
print(pd.DataFrame({
    'libros': [len(pref_train), len(pref_test)],
    'positivos (interes=1)': [int(y_train.sum()), int(y_test.sum())],
    'proporción de 1': [round(y_train.mean(), 2), round(y_test.mean(), 2)],
}, index=['entrenamiento', 'prueba']))

               libros  positivos (interes=1)  proporción de 1
entrenamiento      88                     44             0.50
prueba             23                     12             0.52


## Paso 5 Entrenar dos clasificadores

Configuración fija para los dos modelos: `LogisticRegression(C=1, max_iter=1000, class_weight="balanced")`.

### Modelo A con TF-IDF

In [40]:
def nueva_regresion():
    return LogisticRegression(C=1, max_iter=1000, class_weight='balanced')

vectorizador = TfidfVectorizer()
X_train_tfidf = vectorizador.fit_transform(pref_train['synopsis'])
X_test_tfidf = vectorizador.transform(pref_test['synopsis'])

modelo_a = nueva_regresion()
modelo_a.fit(X_train_tfidf, y_train)
pred_a = modelo_a.predict(X_test_tfidf)

print(f"Matriz TF-IDF entrenamiento: {X_train_tfidf.shape} | prueba: {X_test_tfidf.shape}")
print(f"Predicciones modelo A: {pred_a.tolist()}")

Matriz TF-IDF entrenamiento: (88, 4488) | prueba: (23, 4488)
Predicciones modelo A: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1]


### Modelo B con embeddings

Usamos el mismo modelo multilingüe del TP2 (`paraphrase-multilingual-MiniLM-L12-v2`), con la sinopsis en su texto natural. El modelo de embeddings queda fijo; solo se entrena la regresión logística.

In [41]:
X_train_emb = embeber(pref_train['synopsis'])
X_test_emb = embeber(pref_test['synopsis'])

modelo_b = nueva_regresion()
modelo_b.fit(X_train_emb, y_train)
pred_b = modelo_b.predict(X_test_emb)

print(f"Matriz embeddings entrenamiento: {X_train_emb.shape} | prueba: {X_test_emb.shape}")
print(f"Predicciones modelo B: {pred_b.tolist()}")

Matriz embeddings entrenamiento: (88, 384) | prueba: (23, 384)
Predicciones modelo B: [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0]


## Paso 6 Comprobar las predicciones

In [42]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score

for nombre, pred in [('Modelo A (TF-IDF)', pred_a), ('Modelo B (embeddings)', pred_b)]:
    print(f"=== {nombre} ===")
    if pred.sum() == 0:
        print("El modelo no predijo ningún 1: precisión de la clase 1 = 0 por zero_division=0")
    print(classification_report(y_test, pred, zero_division=0))
    print(pd.DataFrame(confusion_matrix(y_test, pred, labels=[0, 1]),
                       index=['real 0', 'real 1'], columns=['pred 0', 'pred 1']))
    print()

=== Modelo A (TF-IDF) ===
              precision    recall  f1-score   support

           0       0.50      0.09      0.15        11
           1       0.52      0.92      0.67        12

    accuracy                           0.52        23
   macro avg       0.51      0.50      0.41        23
weighted avg       0.51      0.52      0.42        23

        pred 0  pred 1
real 0       1      10
real 1       1      11

=== Modelo B (embeddings) ===
              precision    recall  f1-score   support

           0       0.57      0.73      0.64        11
           1       0.67      0.50      0.57        12

    accuracy                           0.61        23
   macro avg       0.62      0.61      0.61        23
weighted avg       0.62      0.61      0.60        23

        pred 0  pred 1
real 0       8       3
real 1       6       6



In [43]:
def metricas_clase_1(pred):
    return {
        'precisión': round(precision_score(y_test, pred, zero_division=0), 2),
        'recall': round(recall_score(y_test, pred, zero_division=0), 2),
        'F1': round(f1_score(y_test, pred, zero_division=0), 2),
        'aciertos': int((pred == y_test).sum()),
        'errores': int((pred != y_test).sum()),
    }

tabla_pref = pd.DataFrame({'Modelo A (TF-IDF)': metricas_clase_1(pred_a),
                           'Modelo B (embeddings)': metricas_clase_1(pred_b)}).T
display(tabla_pref)

,precisión,recall,F1,aciertos,errores
Modelo A (TF-IDF),0.52,0.92,0.67,12.0,11.0
Modelo B (embeddings),0.67,0.50,0.57,14.0,9.0


Con 23 libros de prueba un solo libro cambia mucho las métricas. Para ver la variación, hacemos validación cruzada repetida sobre los libros de entrenamiento, con la misma configuración. En el modelo A usamos un `Pipeline` para que `TfidfVectorizer` se ajuste dentro de cada partición. No usamos este resultado para elegir parámetros.

In [44]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)

pipe_a = Pipeline([('tfidf', TfidfVectorizer()), ('lr', nueva_regresion())])
f1_cv_a = cross_val_score(pipe_a, pref_train['synopsis'], y_train, cv=cv, scoring='f1')
f1_cv_b = cross_val_score(nueva_regresion(), X_train_emb, y_train, cv=cv, scoring='f1')

print(pd.DataFrame({
    'F1 clase 1 (media)': [f1_cv_a.mean(), f1_cv_b.mean()],
    'F1 clase 1 (desvío)': [f1_cv_a.std(), f1_cv_b.std()],
}, index=['Modelo A (TF-IDF)', 'Modelo B (embeddings)']).round(3))

                       F1 clase 1 (media)  F1 clase 1 (desvío)
Modelo A (TF-IDF)                   0.537                0.101
Modelo B (embeddings)               0.433                0.146


In [45]:
# Dos libros de prueba para analizar: un error (si lo hubo) y un acierto
resultado_test = pref_test[['slug', 'title', 'synopsis']].copy()
resultado_test['real'] = y_test
resultado_test['pred_A'] = pred_a
resultado_test['pred_B'] = pred_b

errores_b = resultado_test[resultado_test['pred_B'] != resultado_test['real']]
aciertos_b = resultado_test[resultado_test['pred_B'] == resultado_test['real']]
casos = pd.concat([errores_b.head(1), aciertos_b[aciertos_b['real'] == 1].head(1)])
for _, fila in casos.iterrows():
    print(f"{fila['title']} | real: {fila['real']} | A: {fila['pred_A']} | B: {fila['pred_B']}")
    print(f"  {fila['synopsis'][:400]}...\n")

Manual del aventurero | real: 1 | A: 1 | B: 0
  ¿Le gustan los viajes insólitos? ¿Le gusta medir su temple enfrentándose a situaciones peligrosas? ¿Desearía vivir aventuras lejos de las comodidades de la civilización? Los consejos del aventurero más famoso de Alemania cuyas aventuras siempre van hasta el límite de lo posible. Cómo comportarse en situaciones peligrosas y difíciles. Más de 1000 consejos y trucos...

Tierra de hombres | real: 1 | A: 1 | B: 1
  Un día de febrero de 1938, el avión pilotado por Antoine de Saint-Exupéry y su amigo André Prévot despega de Nueva York rumbo a Tierra de Fuego. Cargado con exceso de combustible, el aparato se estrella al final de la pista. Superados cinco días de coma y mientras convalece del terrible accidente, Saint-Exupéry escribe «Tierra de hombres» con la perspectiva de quien contempla el mundo desde la sol...



**Análisis de los dos libros:**

- Libro 1 (Error del modelo B — Falso negativo): Manual del aventurero. El lector lo valoró con 1 y el modelo TF-IDF predijo 1, pero el modelo de embeddings predijo 0. La sinopsis está estructurada como un manual de consejos mediante preguntas retóricas y frases como «consejos y trucos», una estructura sintáctica frecuentemente asociada en el espacio de embeddings a textos utilitarios o de autoayuda (que el modelo asoció a desinterés). En cambio, TF-IDF acertó gracias a la presencia explícita de unigramas como «aventurero», «aventuras» y «viajes», fuertemente correlacionados con los gustos positivos del lector.
- Libro 2 (Acierto del modelo B — Verdadero positivo): Tierra de hombres (Antoine de Saint-Exupéry). El lector lo valoró con 1 y ambos modelos predijeron 1 (el modelo B con una probabilidad de 0.81). Es una obra autobiográfica, histórica y de superación humana; el modelo de embeddings captura exitosamente el tono narrativo de memoria y aventura reflexiva, asociándolo con el núcleo de lecturas de no ficción y biografías que concentran el interés del lector.

## Paso 7 Convertir las predicciones en recomendaciones

In [46]:
from sklearn.metrics.pairwise import cosine_similarity

# Verificamos qué columna de predict_proba corresponde a interes=1
print("classes_ modelo A:", modelo_a.classes_, "| classes_ modelo B:", modelo_b.classes_)
col_a = list(modelo_a.classes_).index(1)
col_b = list(modelo_b.classes_).index(1)

ranking = pref_test[['slug', 'title']].copy()
ranking['interes'] = y_test
ranking['prob_A'] = modelo_a.predict_proba(X_test_tfidf)[:, col_a]
ranking['prob_B'] = modelo_b.predict_proba(X_test_emb)[:, col_b]

# Perfil del lector: promedio de los embeddings de los libros con interes=1 de entrenamiento
perfil = X_train_emb[y_train == 1].mean(axis=0, keepdims=True)
ranking['sim_perfil'] = cosine_similarity(perfil, X_test_emb)[0]

def precision_at_5(columna):
    top = ranking.sort_values(columna, ascending=False).head(5)
    return top, top['interes'].sum() / 5

listas = {'TF-IDF + clasificador': 'prob_A', 'Embeddings + clasificador': 'prob_B', 'Similitud con el perfil': 'sim_perfil'}
resumen_p5 = {}
for nombre, columna in listas.items():
    top, p5 = precision_at_5(columna)
    resumen_p5[nombre] = p5
    print(f"=== {nombre} (precision@5 = {p5:.2f}) ===")
    print(top[['title', columna, 'interes']].to_string(index=False))
    print()

azar = y_test.mean()
resumen_p5['Azar (proporción de 1 en prueba)'] = azar
print(f"Positivos en prueba: {y_test.sum()} de {len(y_test)}")
display(pd.Series(resumen_p5, name='precision@5').round(2))

classes_ modelo A: [0 1] | classes_ modelo B: [0 1]
=== TF-IDF + clasificador (precision@5 = 0.40) ===
                    title   prob_A  interes
              Discursos I 0.550768        1
Estrellas en el Annapurna 0.548529        1
     Retorno a Brideshead 0.546470        0
     La boda en Auschwitz 0.544009        0
     Lo que vio la criada 0.544005        0

=== Embeddings + clasificador (precision@5 = 0.80) ===
                                                            title   prob_B  interes
                                                Tierra de hombres 0.807116        1
Querida Mathilda. No veo el momento en que el hombre eche a andar 0.785529        0
                                   Escuela de Ajedrez 4. Superior 0.728896        1
                         Los asesinos matemáticos atacan de nuevo 0.664524        1
                                             La lección de música 0.652627        1

=== Similitud con el perfil (precision@5 = 0.40) ===
                   

TF-IDF + clasificador               0.40
Embeddings + clasificador           0.80
Similitud con el perfil             0.40
Azar (proporción de 1 en prueba)    0.52
Name: precision@5, dtype: float64

**¿Los clasificadores recomendaron mejor que la similitud?:**

Sí, el clasificador basado en embeddings superó ampliamente a todas las alternativas, alcanzando una precision@5 de 0.80 (4 aciertos sobre 5 libros), frente al 0.52 del azar (12 positivos sobre 23 libros en prueba).

Tanto el modelo TF-IDF como la similitud directa con el perfil obtuvieron un rendimiento modesto de 0.40 (quedando incluso por debajo de la línea de base del azar). La razón radica en que la similitud de cosenos respecto al centroide promedia todos los embeddings de interés, dispersando las dimensiones críticas en un espacio denso de 384 variables y recomendando libros por afinidad temática superficial (como Cuentos de mí mismo o Querida Mathilda). La Regresión Logística sobre embeddings, en cambio, aprende un hiperplano discriminativo que penaliza patrones específicos de desinterés y pondera con precisión las características semánticas que realmente definen las elecciones del lector.

## Paso 8 Recomendar libros del catálogo completo

Usamos el modelo B ya entrenado (sin volver a ajustarlo) sobre los libros del catálogo limpio que el lector todavía no vio. Excluimos todos los libros de `preferencias.csv`, incluidas las abstenciones, y descartamos los candidatos que sean otra versión de un libro ya valorado (mismo autor y título muy parecido, con el mismo criterio del paso 1).

In [47]:
vistos = set(preferencias['slug'])
candidatos = catalogo_limpio[~catalogo_limpio['slug'].isin(vistos)].copy()

# Otra versión de un libro valorado: mismo autor y título muy parecido
valorados = catalogo_limpio[catalogo_limpio['slug'].isin(vistos)]
titulos_por_autor = valorados.groupby('authors_str')['title'].apply(list).to_dict()

def es_otra_version(fila):
    for titulo in titulos_por_autor.get(fila['authors_str'], []):
        if fuzz.token_sort_ratio(fila['title'].lower(), titulo.lower()) > 80:
            return True
    return False

otra_version = candidatos.apply(es_otra_version, axis=1)
print(f"Candidatos descartados por ser otra versión de un libro valorado: {otra_version.sum()}")
display(candidatos[otra_version][['title', 'authors_str']])
candidatos = candidatos[~otra_version]

X_cand = embeber(candidatos['synopsis'])
candidatos['puntuacion'] = modelo_b.predict_proba(X_cand)[:, col_b]
top5 = candidatos.sort_values('puntuacion', ascending=False).head(5)

print(f"Candidatos puntuados: {len(candidatos)}\n")
for _, fila in top5.iterrows():
    print(f"{fila['title']} ({fila['authors_str']}) - puntuación {fila['puntuacion']:.3f}")
    print(f"  {fila['synopsis']}\n")

Candidatos descartados por ser otra versión de un libro valorado: 1


,title,authors_str
2182,Ciencia cristiana,Mark Twain


Candidatos puntuados: 8001

La confesión y el perdón (Jean Delumeau) - puntuación 0.969
  La confesión es un rasgo distintivo de la Iglesia católica. Este hecho posee, aparte de sus connotaciones teológicas, unas considerables implicaciones sociales: establece una comunicación especialmente íntima entre el individuo y la institución eclesiástica, lo libera de su culpa, a la vez que lo ata al sacramento. Otorga al individuo libre albedrío para elegir su salvación, a costa de sacrificar su intimidad frente al confesor. Al adoptar la confesión obligatoria anual en el siglo XIII, la Iglesia romana no valoró la avalancha de problemas que iban a desencadenar «las dificultades de la confesión» —la evaluación de las faltas, la apreciación del arrepentimiento, etc. Jean Delumeau describe la vivencia de la confesión en la vida cotidiana: ¿cuál fue el comportamiento real de los confesores?, ¿cómo vivieron los creyentes la obligación de la confesión? Los consejos de escucha benevolente dados a los

**Comentario del lector sobre las cinco recomendaciones:**

Las cinco recomendaciones principales arrojadas por el modelo fueron:

1. La confesión y el perdón (Jean Delumeau) — Puntuación: 0.969
2. Doce pasos y doce tradiciones (Alcohólicos Anónimos) — Puntuación: 0.958
3. Maestros de la República (María Antonia Iglesias) — Puntuación: 0.956
4. Obrar mal, decir la verdad: la función de la confesión en la justicia (Michel Foucault) — Puntuación: 0.949
5. Eutifrón (Platón) — Puntuación: 0.948

**Evaluación del lector:** Cuatro de las cinco recomendaciones resultan de gran interés (80 % de acierto subjetivo): La confesión y el perdón, Maestros de la República, el curso de Foucault y el diálogo de Platón concuerdan notablemente con las inclinaciones del lector por la historia social, la filosofía clásica, el pensamiento crítico y la memoria histórica. La única recomendación desalineada es Doce pasos y doce tradiciones, donde el modelo confundió el vocabulario espiritual y ético con la literatura ensayística reflexiva valorada positivamente.

In [48]:
def resumen_extractivo(texto, n=2):
    partes = oraciones(texto)
    if len(partes) <= n:
        return ' '.join(partes) if partes else texto
    X = TfidfVectorizer().fit_transform(partes)
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, 0)
    centralidad = sim.sum(axis=1)
    elegidas = sorted(np.argsort(centralidad)[-n:])
    return ' '.join(partes[i] for i in elegidas)

for _, fila in top5.iterrows():
    print(f"=== {fila['title']} ===")
    print(f"Resumen: {resumen_extractivo(fila['synopsis'])}\n")

=== La confesión y el perdón ===
Resumen: Al adoptar la confesión obligatoria anual en el siglo XIII, la Iglesia romana no valoró la avalancha de problemas que iban a desencadenar «las dificultades de la confesión» —la evaluación de las faltas, la apreciación del arrepentimiento, etc. Jean Delumeau describe la vivencia de la confesión en la vida cotidiana: ¿cuál fue el comportamiento real de los confesores?, ¿cómo vivieron los creyentes la obligación de la confesión?

=== Doce pasos y doce tradiciones ===
Resumen: Los Doce Pasos y Las Doce Tradiciones, título que trata de forma clara y directa los doce pasos de recuperación del alcoholismo por una parte y las doce tradiciones por las cuales la sociedad de Alcohólicos Anónimos se ha mantenido en el tiempo. Alcohólicos Anónimos (acrónimo AA) es una comunidad de enfermos alcohólicos, no lucrativa, espiritual, que realiza reuniones entre sus miembros con el fin de ayudarse mutuamente para abandonar el alcoholismo mediante terapia de grupo.

**Un caso en el que el resumen funcionó bien y uno en el que perdió información importante:**

- **Caso que funcionó bien:** "Obrar mal, decir la verdad (Michel Foucault)." El método de centralidad de grado sobre la matriz de similitud TF-IDF seleccionó oraciones que sintetizan con precisión la tesis central del libro: la reflexión de Foucault sobre el decir la verdad y la confesión en las instituciones judiciales y religiosas, vinculándolo con su proyecto genealógico de la sexualidad.
- **Caso en el que perdió información importante:** "Eutifrón (Platón)." El algoritmo seleccionó exclusivamente dos viñetas enumerativas («• Segunda definición...» y «• Tercera definición...») por compartir términos centrales repetidos como «justicia» y «dioses», descartando la primera oración que introducía la obra, la acusación de Sócrates y el marco dramático del diálogo, produciendo un resumen fragmentario e incoherente.

In [49]:
ETIQUETAS_NUCLEO = {'NOUN', 'PROPN'}

def frases_y_entidades(sinopsis):
    frases = Counter()
    entidades = Counter()
    for doc in nlp.pipe(sinopsis):
        for chunk in doc.noun_chunks:
            if chunk.root.pos_ in ETIQUETAS_NUCLEO:
                frases[chunk.root.lemma_.lower()] += 1
        for ent in doc.ents:
            entidades[f"{ent.text} ({ent.label_})"] += 1
    return frases, entidades

frases_1, entidades_1 = frases_y_entidades(pref_train[pref_train['interes'] == 1]['synopsis'])
frases_0, entidades_0 = frases_y_entidades(pref_train[pref_train['interes'] == 0]['synopsis'])

def tabla_top(c1, c0, n=10):
    return pd.DataFrame({
        'interes = 1': [f"{k} ({v})" for k, v in c1.most_common(n)],
        'interes = 0': [f"{k} ({v})" for k, v in c0.most_common(n)],
    })

print("Núcleos de frases sustantivas (lema), 10 más frecuentes:")
display(tabla_top(frases_1, frases_0))
print("Entidades nombradas, 10 más frecuentes:")
display(tabla_top(entidades_1, entidades_0))

Núcleos de frases sustantivas (lema), 10 más frecuentes:


,interes = 1,interes = 0
0,vida (34),libro (25)
1,libro (24),vida (24)
2,obra (19),año (16)
3,mundo (16),edición (15)
4,historia (14),tiempo (11)
5,personaje (14),mundo (10)
6,guerra (13),obra (10)
7,siglo (11),antonio (10)
8,año (11),ciencia (9)
9,nancy (10),arte (9)


Entidades nombradas, 10 más frecuentes:


,interes = 1,interes = 0
0,España (LOC) (7),Inglaterra (LOC) (8)
1,Jung (PER) (6),Antonio Pérez (PER) (7)
2,Nancy (PER) (5),Madrid (LOC) (5)
3,Andalucía (LOC) (4),Ivy (ORG) (5)
4,Patricia (PER) (4),España (LOC) (4)
5,Friedman (PER) (4),Felipe II (PER) (4)
6,América (LOC) (4),Polonia (LOC) (3)
7,Este libro (MISC) (4),Gabe (MISC) (3)
8,Nancy (LOC) (3),Novalis (PER) (3)
9,Barcelona (LOC) (3),Kirrin (LOC) (3)


**Comentario del lector sobre si la lista describe lo que le gusta:**

La lista de núcleos sustantivos y entidades nombradas **describe con gran fidelidad el perfil de lectura**:
- En las frases con `interes = 1` destacan lemas como *historia*, *guerra*, *siglo*, *obra* y *personaje*, en contraste con términos más genéricos o circunstanciales en `interes = 0`.
- En las entidades nombradas se evidencia la preferencia por pensadores de peso conceptual (*Jung*, *Friedman*) y contextos históricos y geográficos determinados (*España*, *América*, *Barcelona*), frente a entidades de novelas de ficción o sagas juveniles (*Ivy*, *Kirrin*) que predominan en los libros descartados.

In [50]:
def recomendar(n=5):
    mejores = candidatos.sort_values('puntuacion', ascending=False).head(n)
    return pd.DataFrame({
        'slug': mejores['slug'],
        'title': mejores['title'],
        'puntuacion': mejores['puntuacion'].round(3),
        'resumen': mejores['synopsis'].apply(resumen_extractivo),
    }).reset_index(drop=True)

recomendar(5)

,slug,title,puntuacion,resumen
0,la-confesion-y-el-perdon,La confesión y el perdón,0.969,Al adoptar la confesión obligatoria anual en e...
1,doce-pasos-y-doce-tradiciones,Doce pasos y doce tradiciones,0.958,"Los Doce Pasos y Las Doce Tradiciones, título ..."
2,maestros-de-la-republica,Maestros de la República,0.956,"Diez historias heroicas, ocurridas en distinto..."
3,obrar-mal-decir-la-verdad-la-funcion-de-la-con...,"Obrar mal, decir la verdad: la función de la c...",0.949,Pero Foucault va más allá: reflexiona acerca d...
4,eutifron,Eutifrón,0.948,• Segunda definición de Eutifrón: la justicia ...
